# 第 11 课 · 动态渲染：Selenium 与 Playwright

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 本课只做一件事：搞懂“为什么 `requests` 有时拿到的网页是空的”，并学会让一个**真浏览器**替我们把网页渲染好再取数据。
> 只引入三个新概念：**JS 动态渲染、Selenium、Playwright**。

In [1]:
# 每本 notebook 开头都要重新 import（哪怕上一课写过）
from pathlib import Path                 # 用来拼文件路径
from bs4 import BeautifulSoup            # 用来解析 HTML 文本
import sys

# 本地夹具目录（绝对路径，断网也能跑）
DATA_DIR = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\data')
print('数据目录存在吗：', DATA_DIR.exists())
print('当前内核：', sys.executable)        # 路径里应看到 \envs\ai_learn\python.exe


数据目录存在吗： True
当前内核： C:\Users\Public\miniconda3\envs\ai_learn\python.exe


## 11.1 先看一个“怪事”：网页里明明有商品，requests 却抓不到？

我们的本地夹具 `dynamic_demo.html` 里，`<ul id="item-list">` 这个列表，**文件刚打开时是空的**。
真正的 3 件商品，是页面里一段 JavaScript 在浏览器里跑起来 0.5 秒后才“塞”进去的。

先用上几课学的老办法（直接读 HTML 文本 + BeautifulSoup）抓一次，看看能抓到几条：

In [2]:
# 老办法：把 HTML 当纯文本读进来，找 #item-list 里的 .item
html_text = (DATA_DIR / 'dynamic_demo.html').read_text(encoding='utf-8')
soup = BeautifulSoup(html_text, 'lxml')

items = soup.select('#item-list .item')     # 找所有 class=item 的 li
print('老办法抓到的商品数量：', len(items))
print('→ 是 0 条！因为商品数据是 JS 运行后才生成的，原始 HTML 里根本没有。')


老办法抓到的商品数量： 0
→ 是 0 条！因为商品数据是 JS 运行后才生成的，原始 HTML 里根本没有。


### 这就是“动态渲染”

**生活化类比**：你去餐厅，服务员先给你一张**空菜单**（HTML 壳子），
你坐下后，后厨才现炒、再把菜名一道道补写上去（JavaScript 跑起来填数据）。
`requests` 就像**只抢了那张空菜单就走**，菜还没补上，当然啥都没有。

- **是什么**：动态渲染 = 服务器先吐一个空壳网页，再由浏览器执行 JS 去异步拉数据、填充内容。
- **为什么需要它**：现代网站想“页面加载快、内容能局部刷新不白屏”，所以把数据加载交给浏览器做。
- **对爬虫的麻烦**：你用 `requests` 拿到的只是空菜单。

**用 C++ 类比**：`requests` 拿到的是“编译前的源码字符串”，而你要的是“运行后内存里的结果”；
直接读源码字符串当然看不到运行期才产生的数据。

## 11.2 遇到这种站，有两条路

1. **抓接口（最省）**：打开浏览器开发者工具 Network，找到那段 JS 实际是向哪个网址要的数据（通常返回 JSON），你直接请求那个网址。—— 像直接打电话问后厨“菜单上有啥”。
2. **真浏览器渲染（本课重点）**：让一个**真的浏览器**把页面完整跑一遍（执行 JS、等数据补上），你再从渲染好的页面里取。—— 像真的坐在餐厅里等菜上齐。

浏览器自动化有两个主流工具：**Selenium** 和 **Playwright**。下面都用本地夹具演示，不依赖网络。

## 11.3 Selenium：用代码“遥控”一个真浏览器

**是什么**：Selenium 是一个库，你写 Python 代码，它就像机械手一样去指挥一个真实的 Chrome：打开页面、等它加载、然后把页面上的元素读出来。

**生活化类比**：你远程遥控一个真人帮你坐在电脑前刷网页，他看到什么，你就通过代码读到什么。

下面这一格做四件事：启动一个**不弹窗**的 Chrome → 打开本地页面 → **显式等待**那 3 个商品出现 → 读出它们。

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 把本地 html 变成 file:// 开头的网址，浏览器才能打开
file_url = (DATA_DIR / 'dynamic_demo.html').resolve().as_uri()
print('要打开的网址：', file_url)

selenium_result = None
try:
    opts = Options()
    opts.add_argument('--headless=new')   # 无头模式：不弹出浏览器窗口，后台默默跑
    opts.add_argument('--disable-gpu')

    driver = webdriver.Chrome(options=opts)        # 启动一个受我们控制的 Chrome
    driver.get(file_url)                            # 让它打开本地页面

    # 显式等待：最多等 10 秒，直到页面上真的出现了 #item-list 里的 .item
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, '#item-list .item'))
    )

    # 等待结束后，再读元素（这时 JS 已经把 3 件商品补上去了）
    items = driver.find_elements(By.CSS_SELECTOR, '#item-list .item')
    selenium_result = [it.text for it in items]
    print('【Selenium】渲染后抓到', len(selenium_result), '个商品：')
    for t in selenium_result:
        print('   -', t)
    driver.quit()                                   # 用完一定要关闭浏览器
except Exception as e:
    print('【Selenium 这次没跑起来，不影响学习】', type(e).__name__, e)


要打开的网址： file:///C:/Users/%E5%8D%83%E7%A6%A7/Doubao/chats/2026-09-23/new-chat/AI%E5%AD%A6%E4%B9%A0/08_%E7%88%AC%E8%99%AB/data/dynamic_demo.html


【Selenium】渲染后抓到 3 个商品：
   - 机械键盘 · ¥299
   - 无线鼠标 · ¥89
   - 显示器支架 · ¥159


### 为什么用“显式等待”，而不是 `time.sleep(1)`？

- `time.sleep(0.5)` 是**死等固定时间**：如果网络慢、JS 卡了，0.5 秒后数据还没补上，你读到的还是空的；如果 JS 0.1 秒就补好了，你又白白多等了 0.4 秒。
- `WebDriverWait(...).until(条件)` 是**等到条件满足就立刻继续**：元素一出现马上往下走，既不空等也不提前抢。

> 这是写浏览器自动化最重要的习惯：**永远等“元素出现”这个条件，而不是数秒数。**

## 11.4 Playwright：更现代的浏览器自动化

**是什么**：Playwright（微软出品）和 Selenium 干的是同一件事，但更省心——它自带“自动等待”，API 更简洁，连浏览器都自动帮你装好。

**生活化类比**：Selenium 像老式遥控器，每个动作都要你手动按；Playwright 像语音遥控，你说“把商品给我”，它自己会等商品出现。

### 在 Jupyter 里跑 Playwright 的特殊写法（重要）

Jupyter 内核自己在后台跑着一个事件循环，直接用 Playwright 会报错。
规范里给了一个已验证写法：把 Playwright 的异步代码丢到**一个独立线程 + 一个新的事件循环**里跑。
下面这格先把这个“启动器”定义好（看不懂没关系，复制能用就行）：

In [4]:
# 在 Jupyter 里跑 Playwright 的标准套路：独立线程 + 新事件循环
import threading, asyncio

def run_in_new_thread(coro_factory):
    """把一个异步函数放到新线程里跑，最后把结果返回给我们。"""
    box = {}                                   # 用字典在线程间传递结果
    def worker():
        # Windows 上必须用 ProactorEventLoop，否则 Playwright 起不了浏览器
        loop = asyncio.ProactorEventLoop() if sys.platform == 'win32' else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box['value'] = loop.run_until_complete(coro_factory())
        except BaseException as e:
            box['error'] = e
        finally:
            loop.close()
    t = threading.Thread(target=worker)
    t.start(); t.join()
    if 'error' in box:
        raise box['error']
    return box['value']

print('启动器 run_in_new_thread 已准备好。')


启动器 run_in_new_thread 已准备好。


现在用它启动 Playwright，做和刚才 Selenium 一模一样的事：

In [5]:
playwright_result = None
try:
    from playwright.async_api import async_playwright

    async def pw_main():
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)   # 不弹窗
            page = await browser.new_page()
            await page.goto(file_url)                           # 打开本地页面
            # Playwright 内置自动等待：下面这行会一直等到元素出现
            await page.wait_for_selector('#item-list .item', timeout=10000)
            items = page.locator('#item-list .item')
            n = await items.count()
            texts = [await items.nth(i).text_content() for i in range(n)]
            await browser.close()
        return texts

    playwright_result = run_in_new_thread(pw_main)
    print('【Playwright】渲染后抓到', len(playwright_result), '个商品：')
    for t in playwright_result:
        print('   -', t)
except Exception as e:
    print('【Playwright 这次没跑起来，不影响学习】', type(e).__name__, e)
    print('若缺浏览器，可执行：python -m playwright install chromium')


【Playwright】渲染后抓到 3 个商品：
   - 机械键盘 · ¥299
   - 无线鼠标 · ¥89
   - 显示器支架 · ¥159


## 11.5 小结（记住这三句）

1. **动态渲染**：原始 HTML 是空壳，数据靠 JS 运行后才补上，所以 `requests` 抓到 0 条。
2. **浏览器自动化**：让真浏览器把页面跑一遍，再取数据。Selenium 和 Playwright 二选一即可。
3. **显式等待**：等“元素出现”这个条件，不要 `time.sleep` 死等。

> 提醒：浏览器自动化比 `requests` 慢很多（要起一个浏览器），**只在必须渲染的页面才用它**，而且要守第 13 课的合规红线。

## 11.6 随堂小练习（自己动手，空格里写）

1. 把上面 Selenium 那段的等待条件，改成等待 `.item` 的**第三个**元素出现，看看打印结果有什么变化。
2. 想一想：如果一个动态网页同时也提供了 JSON 接口，你会选“抓接口”还是“上浏览器”？为什么？（写一句注释即可）

In [6]:
# TODO：自己写代码完成上面两题（随堂小练，没有单独答案文件）


## 来源与延伸

- Selenium 官方文档：https://www.selenium.dev/documentation/
- Playwright Python 官方文档：https://playwright.dev/python/docs/intro
- 官方动态练习站（JS 渲染版名言）：https://quotes.toscrape.com/js/
- 下一课：当你要抓成百上千个页面时，手搓 requests 就不够用了，认识爬虫框架 Scrapy。